In [7]:
# ============================================================
# 📈 ADVANCED STOCK MARKET DATA ANALYZER
# PROFESSIONAL STREAMLIT DASHBOARD
# ============================================================

# INSTALL REQUIRED LIBRARIES
!pip install streamlit yfinance ta plotly pyngrok --quiet

# ============================================================
# CREATE STREAMLIT APP
# ============================================================

app_code = """

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import streamlit as st
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go

from ta.trend import SMAIndicator, MACD
from ta.volatility import BollingerBands
from ta.momentum import RSIIndicator

# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="Advanced Stock Analyzer",
    layout="wide"
)

# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown(
    '''
    <style>

    .main {
        background-color: #0e1117;
        color: white;
    }

    .stMetric {
        background-color: #1c1f26;
        padding: 15px;
        border-radius: 12px;
        text-align: center;
    }

    </style>
    ''',
    unsafe_allow_html=True
)

# ============================================================
# TITLE
# ============================================================

st.title("📈 Advanced Stock Market Data Analyzer")

st.markdown(
    "Professional Interactive Financial Dashboard"
)

# ============================================================
# SIDEBAR
# ============================================================

st.sidebar.header("⚙ Dashboard Controls")

ticker = st.sidebar.text_input(
    "Enter Stock Ticker",
    "AAPL"
)

start_date = st.sidebar.date_input(
    "Start Date",
    pd.to_datetime("2020-01-01")
)

end_date = st.sidebar.date_input(
    "End Date",
    pd.to_datetime("2025-01-01")
)

# ============================================================
# DOWNLOAD DATA
# ============================================================

df = yf.download(
    ticker,
    start=start_date,
    end=end_date
)

# FIX MULTI INDEX ISSUE
if isinstance(df.columns, pd.MultiIndex):

    df.columns = df.columns.get_level_values(0)

# RESET INDEX
df.reset_index(inplace=True)

# CLEAN COLUMN NAMES
df.columns = [str(col).lower() for col in df.columns]

# REMOVE NULL VALUES
df.dropna(inplace=True)

# ============================================================
# TECHNICAL INDICATORS
# ============================================================

# DAILY RETURNS
df["daily_return"] = df["close"].pct_change()

# MOVING AVERAGES
df["sma20"] = SMAIndicator(
    close=df["close"],
    window=20
).sma_indicator()

df["sma50"] = SMAIndicator(
    close=df["close"],
    window=50
).sma_indicator()

# RSI
df["rsi"] = RSIIndicator(
    close=df["close"],
    window=14
).rsi()

# MACD
macd = MACD(close=df["close"])

df["macd"] = macd.macd()

df["macd_signal"] = macd.macd_signal()

# BOLLINGER BANDS
bb = BollingerBands(close=df["close"])

df["bb_upper"] = bb.bollinger_hband()

df["bb_lower"] = bb.bollinger_lband()

# ============================================================
# BUY / SELL SIGNAL
# ============================================================

df["signal"] = "HOLD"

df.loc[
    df["sma20"] > df["sma50"],
    "signal"
] = "BUY"

df.loc[
    df["sma20"] < df["sma50"],
    "signal"
] = "SELL"

latest_signal = df["signal"].iloc[-1]

# ============================================================
# KPI METRICS
# ============================================================

current_price = round(
    float(df["close"].iloc[-1]),
    2
)

highest_price = round(
    float(df["high"].max()),
    2
)

lowest_price = round(
    float(df["low"].min()),
    2
)

volatility = round(
    float(df["daily_return"].std() * 100),
    2
)

# ============================================================
# KPI DASHBOARD
# ============================================================

col1, col2, col3, col4, col5 = st.columns(5)

col1.metric(
    "Current Price",
    f"${current_price}"
)

col2.metric(
    "Highest Price",
    f"${highest_price}"
)

col3.metric(
    "Lowest Price",
    f"${lowest_price}"
)

col4.metric(
    "Volatility",
    f"{volatility}%"
)

col5.metric(
    "Signal",
    latest_signal
)

# ============================================================
# 1️⃣ MAIN PRICE CHART
# ============================================================

st.subheader("📈 Stock Price & Moving Averages")

price_fig = go.Figure()

# CANDLESTICK
price_fig.add_trace(
    go.Candlestick(
        x=df["date"],
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name="Candlestick"
    )
)

# SMA20
price_fig.add_trace(
    go.Scatter(
        x=df["date"],
        y=df["sma20"],
        mode="lines",
        name="SMA20",
        line=dict(width=2)
    )
)

# SMA50
price_fig.add_trace(
    go.Scatter(
        x=df["date"],
        y=df["sma50"],
        mode="lines",
        name="SMA50",
        line=dict(width=2)
    )
)

# BOLLINGER UPPER
price_fig.add_trace(
    go.Scatter(
        x=df["date"],
        y=df["bb_upper"],
        mode="lines",
        name="BB Upper",
        line=dict(dash="dot")
    )
)

# BOLLINGER LOWER
price_fig.add_trace(
    go.Scatter(
        x=df["date"],
        y=df["bb_lower"],
        mode="lines",
        name="BB Lower",
        line=dict(dash="dot")
    )
)

price_fig.update_layout(
    template="plotly_dark",
    height=700,
    xaxis_rangeslider_visible=False,
    title=f"{ticker} Price Analysis"
)

st.plotly_chart(
    price_fig,
    use_container_width=True
)

# ============================================================
# 2️⃣ RSI CHART
# ============================================================

st.subheader("📊 RSI Indicator")

rsi_fig = go.Figure()

rsi_fig.add_trace(
    go.Scatter(
        x=df["date"],
        y=df["rsi"],
        mode="lines",
        name="RSI"
    )
)

# OVERBOUGHT LINE
rsi_fig.add_hline(
    y=70,
    line_dash="dash"
)

# OVERSOLD LINE
rsi_fig.add_hline(
    y=30,
    line_dash="dash"
)

rsi_fig.update_layout(
    template="plotly_dark",
    height=350,
    title="Relative Strength Index"
)

st.plotly_chart(
    rsi_fig,
    use_container_width=True
)

# ============================================================
# 3️⃣ MACD CHART
# ============================================================

st.subheader("📉 MACD Indicator")

macd_fig = go.Figure()

macd_fig.add_trace(
    go.Scatter(
        x=df["date"],
        y=df["macd"],
        mode="lines",
        name="MACD"
    )
)

macd_fig.add_trace(
    go.Scatter(
        x=df["date"],
        y=df["macd_signal"],
        mode="lines",
        name="Signal Line"
    )
)

macd_fig.update_layout(
    template="plotly_dark",
    height=350,
    title="MACD Analysis"
)

st.plotly_chart(
    macd_fig,
    use_container_width=True
)

# ============================================================
# 4️⃣ VOLUME ANALYSIS
# ============================================================

st.subheader("📦 Trading Volume")

volume_fig = go.Figure()

volume_fig.add_trace(
    go.Bar(
        x=df["date"],
        y=df["volume"],
        name="Volume"
    )
)

volume_fig.update_layout(
    template="plotly_dark",
    height=350,
    title="Trading Volume"
)

st.plotly_chart(
    volume_fig,
    use_container_width=True
)

# ============================================================
# 5️⃣ DAILY RETURNS DISTRIBUTION
# ============================================================

st.subheader("📈 Daily Returns Distribution")

returns_fig = go.Figure()

returns_fig.add_trace(
    go.Histogram(
        x=df["daily_return"],
        nbinsx=50,
        name="Returns"
    )
)

returns_fig.update_layout(
    template="plotly_dark",
    height=350,
    title="Daily Returns Histogram"
)

st.plotly_chart(
    returns_fig,
    use_container_width=True
)

# ============================================================
# AI INSIGHTS
# ============================================================

st.subheader("🤖 AI Market Insights")

if latest_signal == "BUY":

    st.success(
        "Bullish trend detected based on SMA crossover."
    )

else:

    st.error(
        "Bearish trend detected based on SMA crossover."
    )

if volatility > 3:

    st.warning(
        "High volatility detected. Risk level is elevated."
    )

else:

    st.info(
        "Volatility appears relatively stable."
    )

# ============================================================
# DOWNLOAD REPORT
# ============================================================

csv = df.to_csv(index=False)

st.download_button(
    label="📥 Download CSV Report",
    data=csv,
    file_name=f"{ticker}_analysis.csv",
    mime="text/csv"
)

# ============================================================
# DATASET PREVIEW
# ============================================================

st.subheader("📄 Dataset Preview")

st.dataframe(df.tail(20))

"""

# ============================================================
# SAVE STREAMLIT APP
# ============================================================

with open("app.py", "w") as f:
    f.write(app_code)

# ============================================================
# START NGROK
# ============================================================

from pyngrok import ngrok

# CLEAR OLD TUNNELS
ngrok.kill()

# CREATE PUBLIC URL
public_url = ngrok.connect(8501)

print("🚀 STREAMLIT DASHBOARD RUNNING")
print("🌍 Public URL:")
print(public_url)

# ============================================================
# RUN STREAMLIT
# ============================================================

!streamlit run app.py &>/dev/null &

🚀 STREAMLIT DASHBOARD RUNNING
🌍 Public URL:
NgrokTunnel: "https://haziness-phoniness-playoff.ngrok-free.dev" -> "http://localhost:8501"
